# KOVA3 tutorial 1: stream a gene or interval from the sites-only VCF on Amazon S3

**KOVA3 release:** `v3.0.0` (pinned; see the release manifest)  
**Tier:** Open (no credentials, no AWS account required)  
**Tools:** bcftools 1.19+ (htslib built with libcurl), `pandas` for the table cell, `curl` and the AWS CLI (both optional)

This notebook shows how to fetch only the records you need from the chromosome-sharded, tabix-indexed sites-only VCF without downloading the shard. htslib reads the `.tbi` index over HTTPS and issues range requests for the compressed blocks that cover the interval.

At the end you will see the expected output, the runtime, and the bytes transferred, so you can compare against your own run.

## 0. Parameters

Replace the placeholders once the release is on the Registry of Open Data on AWS. Everything below is driven by these values.

In [ ]:
# Public open-tier bucket and release prefix. The bucket is in ap-northeast-2
# and is not created yet, so these cells will not resolve until launch.
BUCKET  = "kova3-open"
REGION  = "ap-northeast-2"                # bucket region
RELEASE = "v3.0.0"                        # pinned; see docs/versioning.md
BASE    = f"https://{BUCKET}.s3.{REGION}.amazonaws.com/data/release={RELEASE}/sites_vcf"

# Interval to fetch: PCSK9 on GRCh38 as a worked example
CHROM  = "chr1"
REGION_STR = "chr1:55039447-55064852"     # PCSK9 gene body

VCF_URL = f"{BASE}/kova3.{CHROM}.sites.vcf.gz"
print(VCF_URL)

## 1. Check the tools

bcftools must be built with libcurl so that it can open `https://` URLs. The version line shows `libcurl` in the features list when it can.

In [ ]:
!bcftools --version | head -n 2
!bcftools --version | grep -i -o 'libcurl' || echo 'WARNING: bcftools lacks libcurl; install a build with htslib+libcurl (e.g. conda-forge bcftools)'

## 2. Read the header only

This transfers a few kilobytes: the header block plus the index. It is the fastest way to see which INFO fields the release carries.

In [ ]:
import time, subprocess
t0 = time.time()
hdr = subprocess.run(["bcftools", "view", "-h", VCF_URL], capture_output=True, text=True, check=True).stdout
print(f"header fetched in {time.time()-t0:.1f} s, {len(hdr.splitlines())} lines")
print("\n".join(l for l in hdr.splitlines() if l.startswith("##INFO=")))

## 3. Fetch one interval

`-r` uses the tabix index so only the compressed blocks overlapping the interval are requested. `%INFO/...` in the query picks the fields for a compact table.

In [ ]:
t0 = time.time()
cmd = ["bcftools", "query", "-r", REGION_STR,
       "-f", "%CHROM\t%POS\t%REF\t%ALT\t%INFO/AC\t%INFO/AN\t%INFO/AF\t%INFO/nhomalt\t%FILTER\n", VCF_URL]
out = subprocess.run(cmd, capture_output=True, text=True, check=True).stdout
elapsed = time.time()-t0
rows = [l.split("\t") for l in out.strip().splitlines()]
print(f"{len(rows)} variants in {REGION_STR} fetched in {elapsed:.1f} s")
print("\t".join(["CHROM","POS","REF","ALT","AC","AN","AF","nhomalt","FILTER"]))
print("\n".join("\t".join(r) for r in rows[:10]))

## 4. Look at the result as a table

In [ ]:
import pandas as pd

COLUMNS = ["CHROM", "POS", "REF", "ALT", "AC", "AN", "AF", "nhomalt", "FILTER"]
df = pd.DataFrame(rows, columns=COLUMNS)

# An interval with no variants is a legitimate answer, not an error. Guard the
# summary so the cell reports it instead of raising.
if df.empty:
    print(f"no variants returned for {REGION_STR}; nothing to summarize")
else:
    for c in ["POS", "AC", "AN", "nhomalt", "AF"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    print(df[["POS", "AC", "AN", "AF", "nhomalt"]].describe().T[["count", "min", "max"]])

df.sort_values("AF", ascending=False).head(10)

## 5. Same thing from the shell

The equivalent one-liner, for pipelines or a quick look:

In [ ]:
!bcftools view -r {REGION_STR} -H {VCF_URL} | head -n 5

## 6. Bytes transferred and approximate cost

Reading from the open bucket is free for the user: the AWS Open Data Sponsorship Program covers storage and egress for the dataset. The figures below are only to show how little is moved. Run this cell with `curl` to measure the header + index size; the interval query itself requests a handful of 64 KB bgzip blocks.

In [ ]:
!curl -sI {VCF_URL}.tbi | grep -i content-length
!curl -sI {VCF_URL} | grep -i content-length

## Expected output (to be filled at release)

| Item | Value |
|---|---|
| Release | `v3.0.0` |
| Interval | chr1:55039447-55064852 (PCSK9) |
| Variants returned | `<N>` |
| Runtime (header + query) | `<~2-5 s>` on a typical connection |
| Bytes transferred | `<~1-2 MB>` (index + overlapping blocks) |
| User-side AWS cost | none (public bucket, sponsored egress) |

## Next steps

Tutorial 2 annotates your own patient VCF with these frequencies (`bcftools annotate`), tutorial 3 runs the same lookup with Amazon Athena against the Parquet layer, and tutorial 4 loads the prebuilt Hail Table for genome-wide work.